In [9]:
import os

In [10]:
%pwd


'c:\\users\\prasa\\chicken-disease'

In [11]:
os.chdir("c:/users/prasa/chicken-disease")
%pwd

'c:\\users\\prasa\\chicken-disease'

In [12]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir:Path
    tensorboard_dir: Path
    checkpoint_filepath: Path
    

In [13]:
from src.chicken_project.constant import *
from src.chicken_project.utils.common import read_yaml,create_directories
class ConfigurationManager:
    def __init__(self,config_filepath=config_file_path,
                 params_filepath=params_file_path):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        create_directories([self.config.artifacts_roots])

    def get_Preparecallbacksconfig(self)->PrepareCallbacksConfig:
        config=self.config.preparecallbacks
        model_ckpt_dir= os.path.dirname(config.checkpoint_filepath)
        create_directories([Path(model_ckpt_dir),Path(config.tensorboard_dir)])
        Preparecallbacksconfig= PrepareCallbacksConfig(root_dir= Path(config.root_dir),
                                                       tensorboard_dir= Path(config.tensorboard_dir),
                                                       checkpoint_filepath= Path(config.checkpoint_filepath))
        return  Preparecallbacksconfig

In [14]:

from src import logger
from src.chicken_project.utils.common import get_size
import tensorflow as tf
import time


class PrepareCallbacks:
    def __init__(self,config:PrepareCallbacksConfig):
        self.config=config
    @property
    def create_callbacks(self):
        timestamp = time.strftime("%y-%m-%d-%H-%M-%S")
        tb_running_log_dir= os.path.join(self.config.tensorboard_dir,f"tb logs_at{timestamp}",)
        return tf.keras.callbacks.TensorBoard(log_dir= tb_running_log_dir)
    @property
    def create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(filepath=str(self.config.checkpoint_filepath),
                                                  save_best_only=True)
    @property
    def get_tb_ckpt_callbacks(self):
        return [
            self.create_callbacks,
            self.create_ckpt_callbacks
        ]

In [15]:
try:
    config=ConfigurationManager()
    PrepareCallbacks_config= config.get_Preparecallbacksconfig()
    prepare_Callbacks= PrepareCallbacks(config= PrepareCallbacks_config)
    callback_list= prepare_Callbacks.get_tb_ckpt_callbacks
except Exception as e:
    raise e

[2026-05-17 06:57:44,239:INFO:common:yaml file:config\config.yamlloaded successfully]
[2026-05-17 06:57:44,244:INFO:common:yaml file:params.yamlloaded successfully]
[2026-05-17 06:57:44,247:INFO:common:created directory at:artifacts]
[2026-05-17 06:57:44,254:INFO:common:created directory at:artifacts\preparecallbacks\checkpoint_dir]
[2026-05-17 06:57:44,255:INFO:common:created directory at:artifacts\preparecallbacks\tensorboard_dir]
